In [1]:
!pip install pandas scikit-learn joblib

In [2]:
import pandas as pd

# Load the dataset
df = pd.read_csv("track_risk_data.csv")

# Always look at your data first
print(df.shape)        # (100, 10) — rows, columns
print(df.head())        # first 5 rows
print(df.info())        # data types, missing values
print(df["risk_level"].value_counts())   # how many of each category

(100, 10)
  track_id  condition_score  defects  failures  days_since_inspection  \
0     S001               34        0         5                     22   
1     S002               33       10         5                     39   
2     S003               24        0         0                     18   
3     S004               23        8         1                     50   
4     S005               77        9         2                     56   

   traffic_density  track_age  maintenance_duration  risk_score risk_level  
0               51         16                     2   50.116667     MEDIUM  
1               31         39                     4   76.000000   CRITICAL  
2               49         34                     5   41.650000     MEDIUM  
3               89         28                     2   70.783333       HIGH  
4               20         12                     6   46.233333     MEDIUM  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 100 entries, 0 to 99
Data columns (tota

In [4]:
# Features: everything except IDs and the outputs we're trying to predict
features = ["condition_score", "defects", "failures", "days_since_inspection",
            "traffic_density", "track_age", "maintenance_duration"]

X = df[features]
y = df["risk_level"]   # predicting the category: LOW/MEDIUM/HIGH/CRITICAL


In [5]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
# 80% of data to train the model, 20% held back to test it honestly

In [6]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train, y_train)

RandomForestClassifier(random_state=42)

In [7]:
from sklearn.metrics import accuracy_score, classification_report

y_pred = model.predict(X_test)

print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))

Accuracy: 0.9
              precision    recall  f1-score   support

        HIGH       0.88      1.00      0.93         7
         LOW       0.00      0.00      0.00         1
      MEDIUM       0.92      0.92      0.92        12

    accuracy                           0.90        20
   macro avg       0.60      0.64      0.62        20
weighted avg       0.86      0.90      0.88        20



/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [8]:
import pandas as pd

importances = pd.Series(model.feature_importances_, index=features)
print(importances.sort_values(ascending=False))

defects                  0.203987
days_since_inspection    0.164178
condition_score          0.163032
track_age                0.158621
traffic_density          0.122701
failures                 0.122693
maintenance_duration     0.064788
dtype: float64


In [9]:
import joblib

joblib.dump(model, "risk_model.pkl")

['risk_model.pkl']

In [10]:
new_track = pd.DataFrame([{
    "condition_score": 30,
    "defects": 8,
    "failures": 4,
    "days_since_inspection": 55,
    "traffic_density": 90,
    "track_age": 35,
    "maintenance_duration": 3
}])

prediction = model.predict(new_track)
print("Predicted risk level:", prediction[0])

Predicted risk level: HIGH


In [11]:
print("Data shape:", df.shape)
print("Features used:", X.columns.tolist())
print("Train/test sizes:", X_train.shape, X_test.shape)
print("Accuracy:", accuracy_score(y_test, y_pred))
print(classification_report(y_test, y_pred))
print("Feature importances:\n", importances.sort_values(ascending=False))

Data shape: (100, 10)
Features used: ['condition_score', 'defects', 'failures', 'days_since_inspection', 'traffic_density', 'track_age', 'maintenance_duration']
Train/test sizes: (80, 7) (20, 7)
Accuracy: 0.9
              precision    recall  f1-score   support

        HIGH       0.88      1.00      0.93         7
         LOW       0.00      0.00      0.00         1
      MEDIUM       0.92      0.92      0.92        12

    accuracy                           0.90        20
   macro avg       0.60      0.64      0.62        20
weighted avg       0.86      0.90      0.88        20

Feature importances:
 defects                  0.203987
days_since_inspection    0.164178
condition_score          0.163032
track_age                0.158621
traffic_density          0.122701
failures                 0.122693
maintenance_duration     0.064788
dtype: float64


/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.13/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
